### 1. Import Library
Memuat library pandas untuk manipulasi data tabular.

In [ ]:
import pandas as pd
import numpy as np

### 2. Memuat Dataset (Read CSV)
Menggunakan `pd.read_csv` untuk membaca data `sentiment140.csv`. Dataset ini pada umumnya tidak memiliki baris *header* sehingga kita perlu mendefinisikannya secara manual. Parameter `encoding` disesuaikan untuk menangani karakter khusus.

In [ ]:
columns = ['target', 'id', 'date', 'flag', 'user', 'text']
df = pd.read_csv('sentiment140.csv', names=columns, encoding='latin-1')

### 3. Inspeksi Dasar
Mengecek sekilas data menggunakan fungsi `head()`, `tail()`, dan struktur dataframe (`info()` / `shape`).

In [ ]:
display(df.head())
display(df.tail(3))
df.info()
display(df.shape)

### 4. Pengecekan Missing Value (`isna`) & Duplikasi
Menggunakan `isna().sum()` untuk menghitung baris kosong, diikuti proses `dropna` dan drop data duplikat jika ada.

In [ ]:
display(df.isna().sum())
display(df.duplicated().sum())

df = df.dropna()
df = df.drop_duplicates(subset=['text'])
df = df.reset_index(drop=True)

### 5. Hitungan dan Agregasi (`count`, `value_counts`, `groupby`)
Memahami sebaran kelas target, frekuensi data, dan menggabungkan perintah untuk menghitung berdasarkan atribut.

In [ ]:
display(df['target'].value_counts())
display(df.groupby('target').count())

### 6. Operasi String Khusus Teks
Pandas sangat andal untuk pre-processing String. Melalui aksesor `.str.`, kita bisa melakukan lower case, regex replace (seperti menghapus URL atau *username* khusus twitter dsb), pengecekan regex contains, hingga menghitung panjang kata.

In [ ]:
df['text'] = df['text'].str.lower()
df['text'] = df['text'].str.replace(r'http\S+|www\S+', '', regex=True)
df['text'] = df['text'].str.replace(r'@[A-Za-z0-9_]+', '', regex=True)
df['word_count'] = df['text'].str.split().str.len()

display(df[df['text'].str.contains("good|great|happy", na=False)][['target', 'text', 'word_count']].head())

### 7. Transformasi Custom menggunakan `map` dan `apply`
Digunakan jika butuh pemrosesan baris secara lebih kompleks. Map digunakan menukar label integer `sentiment140` (0=negatif, 4=positif) ke string text, sedangkan `.apply` mengaktifkan iterasi *custom function* per sel teks untuk filtering karakter.

In [ ]:
sentiment_map = {0: 'negative', 2: 'neutral', 4: 'positive'}
df['sentiment_label'] = df['target'].map(sentiment_map)

def clean_special_chars(text):
    return ''.join(e for e in str(text) if e.isalnum() or e.isspace())

df['clean_text'] = df['text'].apply(clean_special_chars)
display(df[['target', 'sentiment_label', 'text', 'clean_text', 'word_count']].head())

### 8. Slicing Spesifik, Nilai Unik, dan Filter Terkondisi (`iloc`, `unique`)
Memilih data berdasarkan indeks baris/kolom menggunakan `.iloc`, mengidentifikasi kategori unik lewat `.unique()`, dan melakukan filter baris data sederhana (seperti baris spesifik di pipeline BERT).

In [ ]:
display(df.iloc[:3, :])
display(df['target'].unique())
display(df[df['target'] == 0].head(2))

### 9. Pembuatan DataFrame Baru dan Penggabungan String Kolom
Membuat template tabel DataFrame kosong berisi nama kolom spesifik, lalu diisi menggunakan nilai hasil konkatenasi (perangkaian) dua buah kolom `string` secara langsung.

In [ ]:
df_new = pd.DataFrame(columns=['label', 'combined_text'])
df_new['label'] = df['sentiment_label']
df_new['combined_text'] = df['user'] + " says: " + df['clean_text']

display(df_new.head())

### 10. Downsampling Kombinasi (`min`, `sample`, dan `sort_index`)
Teknik inti yang digunakan di baseline BERT untuk menyeimbangkan Dataset: Hitung nilai terkecil per kelas menggunakan `min()`, manfaatkan `apply(lambda)` dengan `.sample()` untuk memilih baris acak sesuai sampel minimum, lalu gunakan `sort_index()` untuk melihat rapinya distribusi baru secara terurut.

In [ ]:
counts = df_new.groupby(['label']).count()
min_count = counts.min().iloc[0]

df_downsampled = df_new.groupby(['label'], group_keys=False).apply(lambda x: x.sample(n=min_count, random_state=42))
df_downsampled = df_downsampled.reset_index(drop=True)

display(df_downsampled['label'].value_counts().sort_index())

### 11. Penggabungan Vertikal (`concat`) & Exporting Baris (`to_csv`)
Memecah dataframe ke dalam dua bagian terpisah hanya sebagai simulasi (*part1* dan *part2*), menggabungkannya kembali lewat `pd.concat`, parameter `ignore_index=True`, lalu menuliskannya mentah ke `.csv` dalam mode tanpa index buatan.

In [ ]:
df_part1 = df_downsampled.iloc[:10]
df_part2 = df_downsampled.iloc[10:20]

df_combined = pd.concat([df_part1, df_part2], ignore_index=True, sort=False)

df_combined.to_csv('sentiment140_processed.csv', index=False)
display(df_combined.shape)

### 12. Mengganti Nama Kolom (`rename`) dan Casting Tipe Data (`astype`)
Seringkali model atau library `datasets` HuggingFace mensyaratkan nama kolom yang pasti (misal: 'text' dan 'label') serta tipe data integer murni. Fungsi ini sangat sering muncul pada tahap penyiapan pipeline ke PyTorch.

In [ ]:
df_combined = df_combined.rename(columns={'combined_text': 'text'})
df_combined['label_encode'] = df_combined['label'].map({'negative': 0, 'neutral': 1, 'positive': 2})
df_combined['label_encode'] = df_combined['label_encode'].fillna(0).astype('int64')

display(df_combined.dtypes)
display(df_combined.head(2))

### 13. Statistik Deskriptif Numerik (`describe`, `mean`, `max`, `quantile`)
Pada NLP, fungsi ini wajib digunakan pada kolom berisi angka (seperti `word_count` atau panjang kalimat) untuk menentukan argumen parameter `MAX_SEQ_LEN` atau token ideal, agar tidak hanya menebak angka.

In [ ]:
df_combined['length'] = df_combined['text'].str.split().str.len()

display(df_combined['length'].describe())
display(df_combined['length'].max())
display(df_combined['length'].quantile(0.95))

### 14. Label-based Indexing (`loc`) vs Positional (`iloc`)
Penting untuk menghindari *SettingWithCopyWarning* di Pandas dan memilah beda modifikasi berdasarkan **Index/Label baris yang sah** (`loc`) dibandingkan dengan indeks absolut array (`iloc`).

In [ ]:
df_combined.loc[0, 'label'] = 'updated_negative'

display(df_combined.loc[0:2, ['text', 'label']])